READING PDF

In [ ]:
from pypdf import PdfReader

#Loading the PDF

reader = PdfReader("/home/nineleaps/Documents/Python Files/sample_rag_dataset.pdf")

print("Number of pages in the document : ", len(reader.pages))

#Reading text from all the pages

text = ''

for page in reader.pages:
    text += page.extract_text()

print(text)

print("Text Extracted Succesfully!")

CHUNKING (USING lANGCHAIN)

In [ ]:
#checking for text splitter working

from langchain_text_splitters import RecursiveCharacterTextSplitter

print("Working Successfully")

In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size = 100, chunk_overlap = 50)

chunks = splitter.split_text(text)

print(chunks)

print(chunks[0])

EMBEDDINGS

In [ ]:
from sentence_transformers import SentenceTransformer

#choosing a model

model = SentenceTransformer('all-MiniLM-L6-v2')

embeddings = model.encode(chunks)

print(embeddings.shape)

STORING IN VECTOR DATABASE

In [ ]:
import faiss
import numpy as np

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(np.array(embeddings))

CONVERTING USER QUESTION INTO EMBEDDINGS

In [ ]:
query = 'What is embedding?'

query_embedding = model.encode([query])

RETRIEVING SIMILAR CHUNKS

In [ ]:
k = 3

distances, indices = index.search(
    np.array(query_embedding),
    k
)

print(indices)

GETTING RELEVANT CONTEXT

In [ ]:
retrieved_chunks = [chunks[i] for i in indices[0]]

context = "\n".join(retrieved_chunks)

print(context)

CONFIGURING THE GEMINI LLM

In [ ]:
#using gemini AI

import google.generativeai as genai

#Configuring API

genai.configure(api_key = "API_KEY")

model_gemini = genai.GenerativeModel("gemini-2.5-flash")

response = model_gemini.generate_content("Hello")

print(response.text)

SENDING THE CONTEXT TO LLM

In [ ]:
#Creating a prompt for Gemini

prompt = f"""
You are a helpful assistant.

Answer ONLY from the provided context.

Give the Answer in detail.

If the answer is not present in the context,
say:
'I could not find the answer in the document.'

Context:
{context}

Question:
{query}
"""

#Generating response
response = model_gemini.generate_content(
    prompt
)

#Final answer
print(response.text)